# Week 1 — LLM Fundamentals, Real Behaviours & First Chatbot

**What's different from a basic tutorial:**
- Uses the shared `LLMClient` — every call is traced (latency, tokens, cost)
- Self-consistency checking to detect unreliable answers
- Token cost calculator for real budget planning
- Ends with a persistent multi-turn chatbot (Streamlit-ready)

**APIs:** OpenAI `gpt-4o-mini`, Anthropic `claude-haiku-4-5`, Wikipedia REST (no auth)

---
### Agenda
1. Setup — shared utilities
2. Tokens & cost — what your budget actually buys
3. Temperature — the determinism dial
4. Hallucination anatomy — 3 patterns with ground truth verification
5. Self-consistency — detecting unreliable answers automatically
6. Model comparison — same task, two models, scored
7. Chatbot — persistent multi-turn conversation with history

---
## Cell 0 — Setup

In [ ]:
!pip install openai anthropic tiktoken tavily-python python-dotenv --quiet

In [ ]:
import sys, os
from IPython.display import display, Markdown
import os
from dotenv import load_dotenv

In [ ]:

# Load keys from .env file in the same folder
# Your .env file should look like this:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...
#   TAVILY_API_KEY=tvly-...

load_dotenv()

OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY",    "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
TAVILY_API_KEY    = os.getenv("TAVILY_API_KEY",    "")

# Check which keys are loaded
print("API Key Status:")
print(f"  OpenAI   : {'✅ loaded' if OPENAI_API_KEY    else '❌ missing — add to .env'}")
print(f"  Anthropic: {'✅ loaded' if ANTHROPIC_API_KEY else '❌ missing — add to .env'}")
print(f"  Tavily   : {'✅ loaded' if TAVILY_API_KEY    else '❌ missing — get free key at tavily.com'}")

# Create clients
from openai import OpenAI
import anthropic

oai_client    = OpenAI(api_key=OPENAI_API_KEY)
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print()
print("✅  All clients ready!")

In [ ]:
import os
from openai import OpenAI
from anthropic import Anthropic


# ----------------------------------------------------
# 1. OpenAI Call (e.g., GPT-4o)
# ----------------------------------------------------
def call_openai(prompt: str) -> str:
    # Automatically pulls OPENAI_API_KEY from environment
    openai_client = OpenAI()
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content


# ----------------------------------------------------
# 2. Anthropic Claude Call (e.g., Claude 3.5 Sonnet)
# ----------------------------------------------------

'''
def call_claude(prompt: str) -> str:
    # Automatically pulls ANTHROPIC_API_KEY from environment
    claude_client = Anthropic()
    
    response = claude_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.content[0].text

'''

# ----------------------------------------------------
# Execution
# ----------------------------------------------------
if __name__ == "__main__":
    prompt = "Explain gravity in one sentence."
    
    response = call_openai(prompt)
    
    # Create nicely formatted Markdown
    markdown_text = f"""# LLM Response

**Prompt:**  
{prompt}

---

**Response:**

{response}

---

*Powered by GPT-4o-mini • {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*
"""

    # Display rendered Markdown (works best in Jupyter Notebook / IPython)
    display(Markdown(markdown_text))


    #print("\n--- Claude Response ---")
    #
    # 
    # print(call_claude(test_prompt))

In [ ]:
# ── Add shared utils to path ───────────────────────────────────────────────────


# If running from the w01/ folder, shared/ is one level up:
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from shared.utils import (
    LLMClient, Tracer, EvalFramework, Guardrails, banner, section, observe, discuss, compare
)
import json, requests, textwrap
import tiktoken

# ── Clients ────────────────────────────────────────────────────────────────────
# Set env vars:  OPENAI_API_KEY  /  ANTHROPIC_API_KEY
# Or paste temporarily (never commit to git)
tracer = Tracer(session_id="w01")
client = LLMClient(tracer=tracer)

GPT    = "gpt-5.0"
HAIKU  = "claude-haiku-4-5"

print("✅  Setup complete — tracer session:", tracer.session_id)

---
## Section 1 — Tokens & Cost: What Your Budget Actually Buys

**Trainer:** Most teams underestimate token costs in production.  
This section makes the invisible visible — every call now has a price tag.

In [ ]:
# ── 1a. Token visualiser ──────────────────────────────────────────────────────
banner("Token visualisation — what the model actually reads")

enc = tiktoken.encoding_for_model("gpt-4o-mini")

samples = [
    "नमस्ते — IT हेल्पडेस्क पर आपका स्वागत है",   # Hindi — 3x tokens per word
    "IT Helpdesk welcomes you",  # English same meaning as ^^
    "Der IT-Helpdesk heißt Sie willkommen", #German, same meaning as ^
    "2024-11-07T14:30:00Z",                          # ISO timestamp
    "supercalifragilisticexpialidocious",
    "P1 SLAR breach RTO/RPO CMDB CI SLA",            # acronyms
]

section("Token counts by text type")
for text in samples:
    tokens  = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"Input  : {text!r}")
    print(f"Tokens : {decoded}")
    print(f"Count  : {len(tokens)} tokens | {len(text)} chars | ratio={len(tokens)/max(len(text.split()),1):.1f} tok/word")
    print()

In [ ]:
# ── 1b. Cost calculator — real budget planning ─────────────────────────────────
banner("Cost calculator — real production estimates")

# Realistic workload: IT helpdesk chatbot
scenarios = [
    {
        "name": "Ticket classifier (ServiceNow auto-routing)",
        "avg_input_tokens": 200,    # ticket description
        "avg_output_tokens": 80,    # JSON classification
        "daily_volume": 500,
        "model": GPT,
    },
    {
        "name": "Chatbot response (employee helpdesk)",
        "avg_input_tokens": 800,    # conversation history + question
        "avg_output_tokens": 300,
        "daily_volume": 1000,
        "model": GPT,
    },
    {
        "name": "Policy summariser (RAG over docs)",
        "avg_input_tokens": 3000,   # retrieved chunks + question
        "avg_output_tokens": 400,
        "daily_volume": 200,
        "model": HAIKU,
    },
]

print(f"{'Scenario':<45} {'Daily calls':>12} {'Daily cost $':>13} {'Monthly $':>10}")
print("─" * 85)
for s in scenarios:
    daily_cost = client.estimate_cost(
        prompt="x" * s["avg_input_tokens"],  # proxy for token count
        expected_output_tokens=s["avg_output_tokens"],
        model=s["model"]
    ) * s["daily_volume"]
    monthly = daily_cost * 30
    print(f"{s['name']:<45} {s['daily_volume']:>12,} {daily_cost:>13.4f} {monthly:>10.2f}")

print()
observe("The summariser (RAG) is cheapest per-call but most expensive in total because of large contexts. "
        "Context window management is a real cost optimisation lever.")

---
## 🎲 Section 2 — How LLMs Predict the Next Word

An LLM doesn't "know" the answer. It **predicts the most likely next token**, over and over,
until it decides to stop.

Think of it like autocomplete on your phone — but trained on most of the internet.

```
You type:   "The capital of France is ..."
Model sees: What token most likely follows this sequence?
Answer:     "Paris" (very high probability)
            "Lyon"  (low probability)
            "blue"  (almost zero probability)
```

The **temperature** setting controls how adventurous the model is:
- `temperature=0` → always picks the most likely token (deterministic)
- `temperature=1` → samples proportionally from likely tokens (creative)
- `temperature=2` → samples from even unlikely tokens (chaotic)

In [ ]:
# ── Visualise "next word prediction" with logprobs ────────────────────────────
# OpenAI lets us see the top 5 tokens it considered at each step.
# This makes the prediction process VISIBLE.

import math

def show_next_word_probabilities(prompt, n_tokens=5):
    """
    Complete the prompt and show the top 5 candidate tokens
    the model considered at the FIRST prediction step.
    """
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,           # only predict ONE token
        logprobs=True,          # show probabilities
        top_logprobs=5,         # show top 5 candidates
        temperature=0.1
    )

    top5 = response.choices[0].logprobs.content[0].top_logprobs

    print(f"📝 Prompt: \"{prompt}\"")
    print(f"\n   Top {len(top5)} candidates for the NEXT word:")
    print(f"   {'Token':<20} {'Probability':>12} {'Bar'}")
    print(f"   {'─'*20} {'─'*12} {'─'*25}")

    for lp in top5:
        prob    = math.exp(lp.logprob) * 100      # convert log-prob to %
        bar     = "█" * int(prob / 2)              # visual bar
        token   = repr(lp.token)                   # show spaces and newlines clearly
        print(f"   {token:<20} {prob:>11.1f}%  {bar}")
    print()


print("=" * 60)
print("NEXT WORD PROBABILITY VISUALISER")
print("=" * 60)
print()

# Unambiguous — one clear answer
show_next_word_probabilities("The capital of France is")

# Slightly ambiguous
show_next_word_probabilities("Python is a programming")

# Very ambiguous — model has to guess
show_next_word_probabilities("The best way to fix a broken")

print("🔑 KEY LESSON:")
print("   When the model is CERTAIN → one token has ~99% probability (Paris)")
print("   When the model is UNCERTAIN → probability spreads across many tokens")
print("   Hallucination happens when the model picks a confident-sounding token")
print("   even when it has no real knowledge to back it up.")

In [ ]:
banner("Temperature experiment — 3 tasks that actually show the difference")

AMBIGUOUS_TICKET = """
After the Windows 11 feature update, several staff say they cannot reach
internal SharePoint sites. Some user can access them via mobile data but not using 
office Wi-Fi. The IT team isn't sure if this is a DNS issue, a proxy config
change from the update, or a SharePoint permission problem, user access problem.

NOTE: This issue equally involves corporate network routing, updated operating system 
software, and user permission access tokens. It is an equal mix of all three.
"""

CLASSIFIER_SYSTEM = """
You are an IT incident classifier. 
Analyze the ticket in one brief sentence, then classify it into ONE category:
[Network, Hardware, Software, Access, Data, Compliance, Other]

Format your output exactly like this:
Analysis: <one sentence>
Category: <category name>
"""

answers_0 = []
print("--- Temperature: 0.0 ---")
for i in range(5):
    ans = client.chat(GPT, user=AMBIGUOUS_TICKET, system=CLASSIFIER_SYSTEM,
                      temperature=0.0, tags=["temp_demo"])
    # Extract only the Category line
    category = next((line.split(":")[-1].strip() for line in ans.splitlines() if line.startswith("Category:")), ans.strip())
    answers_0.append(category)
    print(f"  Run {i+1}: {category}")
print(f"  Unique: {set(answers_0)}\n")

answers_15 = []
print("--- Temperature: 1.5 ---")
for i in range(5):
    ans = client.chat(GPT, user=AMBIGUOUS_TICKET, system=CLASSIFIER_SYSTEM,
                      temperature=1.5, tags=["temp_demo"])
    category = next((line.split(":")[-1].strip() for line in ans.splitlines() if line.startswith("Category:")), ans.strip())
    answers_15.append(category)
    print(f"  Run {i+1}: {category}")
print(f"  Unique: {set(answers_15)}")

In [ ]:

# ── Task 2: Creative generation — where temperature SHOULD be high ───────────
print("=== Task 2: CREATIVE task — tagline for an IT helpdesk product ===")
CREATIVE_PROMPT = "Write one memorable marketing tagline for an AI-powered IT helpdesk product. Max 20 words."

section("temp=0.0 — 5 runs (should be identical)")
taglines_0 = []
for i in range(5):
    ans = client.chat(GPT, user=CREATIVE_PROMPT, temperature=0.0, tags=["temp_demo"])
    taglines_0.append(ans.strip())
    print(f"  Run {i+1}: {ans.strip()}")
print(f"  Unique: {len(set(taglines_0))}/5 — {('all identical' if len(set(taglines_0))==1 else 'some variation')}")

section("temp=1.5 — 5 runs (should vary usefully)")
taglines_1 = []
for i in range(5):
    ans = client.chat(GPT, user=CREATIVE_PROMPT, temperature=1.5, tags=["temp_demo"])
    taglines_1.append(ans.strip())
    print(f"  Run {i+1}: {ans.strip()}")
print(f"  Unique: {len(set(taglines_1))}/5 — {('all different' if len(set(taglines_1))==5 else 'some repeats')}")

observe(
    "temp=0 on a creative task → same tagline 5 times → useless for ideation.\n"
    "temp=1.4 → 5 different options to choose from.\n"
    "Rule: LOW temp for decisions (classification, routing, extraction).\n"
    "      HIGH temp for generation (brainstorming, drafting, creative writing)."
)
print()


In [ ]:

# ── Task 3: Open-ended summary — shows FORMAT drift at high temp ─────────────
print("=== Task 3: FORMAT drift — same summary task, temp=0 vs temp=1.5 ===")
SUMMARY_PROMPT = """
Summarise this IT incident in 2-3 sentences:
'At 2 PM the entire finance team lost access to the SAP system. The root cause was
an expired SSL certificate on the load balancer. An engineer renewed it at 2:48 PM
and access was restored by 3:05 PM. Total downtime: 65 minutes. 45 users affected.'
"""

section("temp=0.0 — 3 runs (check for format consistency)")
for i in range(3):
    ans = client.chat(GPT, user=SUMMARY_PROMPT, temperature=0.0, tags=["temp_demo"])
    words = len(ans.split())
    print(f"  Run {i+1} ({words} words): {ans.strip()[:200]}")

section("temp=1.5 — 3 runs (watch for format/length instability)")
for i in range(3):
    ans = client.chat(GPT, user=SUMMARY_PROMPT, temperature=1.5, tags=["temp_demo"])
    words = len(ans.split())
    print(f"  Run {i+1} ({words} words): {ans.strip()[:200]}")

observe(
    "At temp=1.5 the WORD COUNT and STRUCTURE vary — sometimes bullet points,\n"
    "sometimes prose, sometimes one sentence, sometimes four.\n"
    "In a pipeline that parses the output (e.g. to fill a ServiceNow field),\n"
    "this format drift breaks your code even when the content is correct."
)
discuss("Which of these three tasks maps to something the team actually does?")

In [ ]:
# ── 2b. Character-counting trap ─────────────────────────────────────────────
# Proves tokenisation is the root cause — models can't count characters reliably.
banner("Character counting trap — root cause: tokenisation")

tricky = [
    ("How many letter 'e's are in 'telepresence'?", "5"),
    ("What is the 7th character in 'unbelievable'?", "i"),
    ("Is 'refrigerator' more vowels or consonants?", "consonants (7 vs 5)"),
]

print(f"{'Question':<48} {'GPT says':<25} {'Ground truth':<20} {'Correct?'}")
print("─" * 100)
for q, truth in tricky:
    gpt_ans = client.chat(GPT, user=q, system="Answer in one word or number.", temperature=0).strip()
    correct = "✅" if truth.split()[0].lower() in gpt_ans.lower() else "❌"
    print(f"{q:<48} {gpt_ans:<25} {truth:<20} {correct}")

observe("The model reasons about tokens, not characters. "
        "Character-level tasks (string parsing, formatting, counting) must be done in code.")

---
## Section 3 — Hallucination Anatomy

Three patterns — all demonstrated against Wikipedia ground truth.

In [ ]:
# ── Wikipedia ground truth helper ──────────────────────────────────────────────
def wiki(topic: str, chars: int = 400) -> str:
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic.replace(' ','_')}"
    r = requests.get(url, headers={"User-Agent": "TrainingDemo/1.0"}, timeout=5)
    if r.status_code == 200:
        return r.json().get("extract", "Not found")[:chars]
    return f"Wikipedia returned {r.status_code}"

print(wiki("ServiceNow"))

In [ ]:
# ── Hallucination 1: Fabricated specifics ──────────────────────────────────────
banner("Hallucination Type 1 — Fabricated facts about a real company")

questions = [
    ("When was ServiceNow founded and by whom?",       "ServiceNow"),
    ("Who is the current CEO of Infosys?",              "Infosys"),
    ("What was Wipro's revenue in their last fiscal year?", "Wipro"),
]

for q, wiki_topic in questions:
    gpt_ans  = client.chat(GPT,   user=q, temperature=0).strip()
    #claud_ans = client.chat(HAIKU, user=q, temperature=0).strip()
    wk = wiki(wiki_topic, chars=500)
    print(f"Q: {q}")
    print(f"  GPT  : {gpt_ans[:500]}")
    #print(f"  Haiku: {claud_ans[:120]}")
    print(f"  Wiki : {wk[:200]}")
    print()

In [ ]:
# ── Hallucination 2: Invented citations ────────────────────────────────────────
# Most dangerous pattern in professional/academic work.
banner("Hallucination Type 2 — Invented academic citations")

citation_prompt = """
Give me 3 peer-reviewed papers on ROI of IT service management automation.
Include: author, title, journal, year, DOI.
"""

gpt_citations = client.chat(GPT, user=citation_prompt, temperature=0)
print("GPT citations:")
print(gpt_citations)
print()
print("⚠️  LIVE DEMO: Copy the first DOI above → go to doi.org → see if it exists.")
print("   In most runs, at least 1 of 3 DOIs leads to a 404 or completely different paper.")

In [ ]:
# ── Hallucination 3: Self-knowledge inconsistency ─────────────────────────────
# Models disagree about their OWN training cutoff.
banner("Hallucination Type 3 — Models disagree about themselves")

self_q = "What is your exact training data cutoff date?"

section("GPT-4o-mini (5 runs at temp=0.7)")
gpt_answers = []
for i in range(5):
    ans = client.chat(GPT, user=self_q, temperature=0.7).strip()[:80]
    gpt_answers.append(ans)
    print(f"  Run {i+1}: {ans}")

section("Claude Haiku (5 runs at temp=0.7)")
for i in range(5):
    ans = client.chat(HAIKU, user=self_q, temperature=0.7).strip()[:80]
    print(f"  Run {i+1}: {ans}")

observe("Even at temp=0, models give inconsistent dates about themselves.\n"
        "  If a model is wrong about its own training, what does that tell us about its other facts?")

---
## Section 4 — Self-Consistency: Detecting Unreliable Answers Automatically

**Trainer context:** This is a production technique — not just a demo.  
For critical decisions, run the same prompt 3–5× and flag answers with low agreement.

In [ ]:
# ── 4a. Self-consistency check ────────────────────────────────────────────────
banner("Self-consistency — automatic reliability scoring")

PRIORITY_SYSTEM = """
You are an IT incident prioritiser for a financial services company.
Assign exactly one priority: P1 (critical), P2 (high), P3 (medium), P4 (low).
Return ONLY the priority label.
"""

test_tickets = [
    # Unambiguous cases
    "Payment gateway down. All online transactions failing. Revenue impact: £12,000/min.",
    "My mouse double-clicks when I single-click.",
    # Ambiguous cases — self-consistency will flag these
    "Some users can't access the report portal. Not sure how many. Happened this morning.",
    "The backup job failed last night. No one noticed until now. Data from yesterday is at risk.",
]

RUNS = 5
RELIABILITY_THRESHOLD = 0.6

print(f"{'Ticket':<60} {'Majority':>8} {'Agreement':>10} {'Reliable?':>10}")
print("─" * 95)

for ticket in test_tickets:
    majority, agreement = Guardrails.self_consistency_check(
        client, GPT, PRIORITY_SYSTEM, ticket, n=RUNS
    )
    reliable = "✅" if agreement >= RELIABILITY_THRESHOLD else "⚠️  FLAG"
    print(f"{ticket[:59]:<60} {majority.upper():>8} {agreement:>10.0%} {reliable:>10}")

print()
observe("Low-agreement answers need human review or more context.\n"
        "  High agreement ≠ correct — but low agreement = unreliable.")

In [ ]:
# ── 4b. First evaluation run — scored against expected answers ────────────────
banner("Evaluation framework — Week 1 baseline")

ef = EvalFramework(client, model=GPT, judge_model=GPT)

# Priority classification test cases
ef.add_case("t01", input="Payment gateway down. All transactions failing.", expected="P1", check="contains")
ef.add_case("t02", input="My mouse double-clicks sometimes.", expected="P4", check="contains")
ef.add_case("t03", input="Our entire Bangalore office lost internet.", expected="P1", check="contains")
ef.add_case("t04", input="Excel crashes when I open a specific file.", expected="P3", check="contains")
ef.add_case("t05", input="Need Adobe Acrobat installed on my laptop.", expected="P4", check="contains")

results = ef.run(system_prompt=PRIORITY_SYSTEM)
ef.report(results)

print()
print("💡  Keep this baseline. Week 2 will run the same eval after adding few-shot examples.")
print("   The score difference proves that examples beat instructions.")

---
## Section 5 — Model Comparison: Scored, Not Just Printed

In [ ]:
# ── 5. Structured model comparison ────────────────────────────────────────────
# Two real tasks: (a) instruction following stress test, (b) ambiguous communication
banner("Model comparison — instruction following stress test")

INCIDENT_TEXT = """
At 14:32 Monday, the entire Bangalore office (approx. 300 staff) lost access to SAP.
Root cause: an expired SSL certificate on the load balancer — not flagged in the cert tool.
Engineer Arjun Menon renewed it at 15:10. Services restored by 15:18. Downtime: 46 minutes.
Financial impact estimated at ₹4.2 lakhs in lost productivity.
Certificate tool now configured to alert 30 days before expiry.
"""

STRICT_RULES = """
Summarise this IT incident report. STRICT RULES:
- Exactly 3 bullet points
- Each bullet starts with a tag: [Impact], [Cause], or [Resolution]
- Do NOT use the word 'issue' or 'problem'
- End with a single line: Priority: HIGH or Priority: MEDIUM
- Total response under 100 words
"""

def score_response(response: str) -> dict:
    """Rule-based scoring of the response against the strict rules."""
    r = response.strip()
    bullets    = [l for l in r.splitlines() if l.strip().startswith("-") or l.strip().startswith("•")]
    has_tags   = all(any(tag in b for tag in ["[Impact]","[Cause]","[Resolution]"]) for b in bullets)
    no_issue   = "issue" not in r.lower() and "problem" not in r.lower()
    has_prio   = "Priority: HIGH" in r or "Priority: MEDIUM" in r
    word_count = len(r.split())
    under_100  = word_count <= 100
    return {
        "bullet_count":  len(bullets),
        "correct_tags":  has_tags,
        "no_banned_word": no_issue,
        "has_priority":  has_prio,
        "under_100_words": under_100,
        "word_count":    word_count,
        "score": sum([len(bullets)==3, has_tags, no_issue, has_prio, under_100]) / 5
    }

for model_id, label in [(GPT, "GPT-4o-mini"), (HAIKU, "Claude Haiku")]:
    response = client.chat(model_id, user=STRICT_RULES + "\n\n" + INCIDENT_TEXT, temperature=0)
    scores   = score_response(response)
    print(f"\n── {label} (score: {scores['score']:.0%}) ──")
    print(textwrap.indent(response.strip(), "  "))
    print(f"  Rules: bullets={scores['bullet_count']}/3 | tags={scores['correct_tags']} | "
          f"no_banned={scores['no_banned_word']} | priority={scores['has_priority']} | "
          f"words={scores['word_count']}/100")

observe("Which rules were dropped? The dropped rules are the ones you must enforce with JSON mode (Week 2).")

---
## Section 6 — Chatbot: Persistent Multi-Turn with History

**Trainer context:** This is the first chatbot in the 4-week progression.  
It maintains conversation history properly — the most common mistake beginners make is losing context.

In [ ]:
# ── 6a. Conversation history class ────────────────────────────────────────────
banner("Chatbot — multi-turn IT helpdesk assistant")

class ChatSession:
    """
    Manages conversation history for a multi-turn chatbot.
    Implements a sliding window to keep context within token limits.
    """

    def __init__(self, system: str, model: str, client: LLMClient,
                 max_history_turns: int = 10, temperature: float = 0.3):
        self.system            = system
        self.model             = model
        self.client            = client
        self.max_history_turns = max_history_turns
        self.temperature       = temperature
        self.history: list     = []   # list of {role, content} dicts
        self.turn_count        = 0

    def chat(self, user_message: str) -> str:
        self.turn_count += 1
        self.history.append({"role": "user", "content": user_message})

        # Sliding window — keep system + last N turns
        window = self.history[-(self.max_history_turns * 2):]

        messages = [
            {"role": "system", "content": self.system},
            *window
        ]

        # Estimate context size
        context_tokens = sum(
            self.client.count_tokens(m["content"], self.model)
            for m in messages
        )

        response = self.client.chat(
            self.model, user=user_message, system=self.system,
            temperature=self.temperature,
            messages=messages,
            tags=["chatbot", f"turn_{self.turn_count}"]
        )

        self.history.append({"role": "assistant", "content": response})
        return response, context_tokens

    def show_history(self) -> None:
        print(f"\n{'─'*60}  Conversation history ({self.turn_count} turns)")
        for msg in self.history:
            role  = "👤 User" if msg["role"] == "user" else "🤖 Bot "
            print(f"  {role}: {msg['content'][:120]}")

    def reset(self) -> None:
        self.history = []
        self.turn_count = 0


# ── 6b. Run the chatbot ────────────────────────────────────────────────────────
HELPDESK_SYSTEM = """
You are an IT helpdesk assistant for a technology company in Bangalore.
You help employees with IT issues: software, hardware, network, access, and data problems.
Keep responses concise and actionable. Ask one clarifying question at a time if needed.
If the issue sounds like a P1, say so clearly and tell the user to also call the helpdesk line.
You do not have access to real systems — you can guide users through steps, not take actions.
"""

bot = ChatSession(
    system=HELPDESK_SYSTEM,
    model=GPT,
    client=client,
    max_history_turns=8,
    temperature=0.3
)

# Simulate a realistic 4-turn conversation
conversation = [
    "Hi, my Outlook is not receiving new emails since this morning.",
    "Yes, I can send emails but nothing is arriving. My colleagues say they sent me things.",
    "I'm on Outlook 2021, Windows 11. I restarted already.",
    "Also, while we're at it — my VPN drops every 30 minutes too.",
]

print("Simulating a 4-turn helpdesk conversation...\n")
for user_msg in conversation:
    print(f"👤 User : {user_msg}")
    response, ctx_tokens = bot.chat(user_msg)
    print(f"🤖 Bot  : {response.strip()[:300]}")
    print(f"   [context: ~{ctx_tokens} tokens]")
    print()

bot.show_history()

In [ ]:
# ── 6c. Context window problem — what happens without history ─────────────────
# This is the most common beginner mistake.
banner("Common mistake — stateless calls (no history)")

print("Stateless (each call knows nothing about previous turns):")
print()

turns_stateless = [
    "My Outlook is not receiving emails.",
    "Yes, I already told you — I can send but not receive.",   # model has no context
    "As I said, I restarted twice.",
]

for turn in turns_stateless:
    print(f"👤 User : {turn}")
    # No history passed — each call is fresh
    resp = client.chat(GPT, user=turn, system=HELPDESK_SYSTEM, temperature=0.3)
    print(f"🤖 Bot  : {resp.strip()[:200]}")
    print()

observe("The bot repeats itself, asks questions the user already answered, "
        "and loses all diagnostic progress.\n"
        "  This is the most common production bug in naive LLM integrations.")

In [ ]:
# ── 6d. Session summary — tracer report ───────────────────────────────────────
banner("Session cost and performance summary")
tracer.summary()

print()
print("Export traces to JSONL for analysis:")
tracer.export_jsonl("/tmp/w01_traces.jsonl")

---
## Streamlit Chatbot (Bonus — run as a web app)

Save the cell below as `chatbot_w01.py` and run:
```bash
streamlit run chatbot_w01.py
```

In [ ]:
# ── Streamlit chatbot — save as chatbot_w01.py and run separately ─────────────
streamlit_code = '''
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

import streamlit as st
from shared.utils import LLMClient, Tracer

SYSTEM = """
You are an IT helpdesk assistant. Help employees with IT issues.
Keep responses concise. Ask one clarifying question at a time.
"""

st.set_page_config(page_title="IT Helpdesk Bot — W1", page_icon="🖥️")
st.title("🖥️  IT Helpdesk Chatbot")
st.caption("Week 1 — Basic multi-turn chatbot with token tracking")

# ── Session state ──────────────────────────────────────────────────────────────
if "history" not in st.session_state:
    st.session_state.history = []
if "token_count" not in st.session_state:
    st.session_state.token_count = 0

tracer = Tracer(session_id="streamlit-w01")
client = LLMClient(tracer=tracer)

# ── Sidebar metrics ────────────────────────────────────────────────────────────
with st.sidebar:
    st.metric("Turns", len(st.session_state.history) // 2)
    st.metric("Total cost", f"${tracer.total_cost():.5f}")
    if st.button("Reset conversation"):
        st.session_state.history = []
        st.rerun()

# ── Chat history display ───────────────────────────────────────────────────────
for msg in st.session_state.history:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Input ──────────────────────────────────────────────────────────────────────
if prompt := st.chat_input("Describe your IT issue..."):
    st.session_state.history.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

    messages = [
        {"role": "system", "content": SYSTEM},
        *st.session_state.history[-16:]   # last 8 turns
    ]
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = client.chat(
                "gpt-4o-mini", user=prompt, system=SYSTEM,
                messages=messages, temperature=0.3, tags=["streamlit"]
            )
        st.write(response)

    st.session_state.history.append({"role": "assistant", "content": response})
    st.rerun()
'''

with open("/tmp/chatbot_w01.py", "w") as f:
    f.write(streamlit_code.strip())

print("Streamlit chatbot saved to /tmp/chatbot_w01.py")
print("Run: streamlit run /tmp/chatbot_w01.py")

---
## Week 1 — Key Takeaways

| Topic | What we proved | Production implication |
|---|---|---|
| Tokens | Hindi costs 3× more tokens than English | Budget by token, not by word |
| Temperature | temp=0 for classifiers, high for ideation | Never use temp>0.5 in routing pipelines |
| Hallucination | 3 patterns, all verified with live Wikipedia data | Every fact from an LLM needs a verification strategy |
| Self-consistency | Ambiguous tickets get flagged automatically | Use n=3 consistency check before auto-routing high-stakes decisions |
| Chatbot | History = state; stateless calls break continuity | Always pass full conversation history — not just the last message |
| Tracer | Every call recorded with latency + cost | You cannot optimise what you don't measure |

**Next week:** We control the model's output reliably with structured prompt engineering —  
the same eval cases will run again so you can see the improvement.